In [43]:
# !pip install --upgrade pip setuptools wheel
# !pip install ultralytics opencv-python

In [44]:
import os
import cv2
import torch
import random 
import numpy as np
from tqdm import tqdm
import torch.nn as nn
from ultralytics import YOLO
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import Dataset, DataLoader, random_split

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [46]:
CHARS = list(
  '!"#$%&'+
   "()*+,-./0123456789:<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ^_`abcdefghijklmnopqrstuvwxyz~§µ×ßäöüΩ"
)

# Reserve 0 for CTC blank; labels start at 1
char2idx = {c: i+1 for i, c in enumerate(CHARS)}  # labels start at 1
idx2char = {i: c for c, i in char2idx.items()}

def text_to_labels(text):
    return [char2idx[c] for c in text if c in char2idx]

In [47]:
class WordImageDataset(Dataset):
    def __init__(self, image_dir, label_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.samples = []
        with open(label_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 2:
                    img, label = parts
                    self.samples.append((img, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('L')
        if self.transform:
            image = self.transform(image)
        # Capture width AFTER transform (this is what model sees)
        resized_width = image.size(2)  # tensor: (C, H, W)
        label_encoded = torch.tensor(text_to_labels(label), dtype=torch.long)
        return image, label_encoded, resized_width  


In [48]:
class ResizeKeepAspectRatio(): # Resize to a fixed height, maintaining aspect ratio
    def __init__(self, height):
        self.height = height

    def __call__(self, img):
        width, height = img.size
        new_width = int(width * (self.height / height))
        new_width = max(1, new_width)
        return img.resize((new_width, self.height), Image.LANCZOS)

In [49]:
IMG_HEIGHT = 32
SPLIT_RATIO = 0.8
BATCH_SIZE = 128
EPOCHS = 3 

In [50]:
transform = transforms.Compose([
    ResizeKeepAspectRatio(IMG_HEIGHT), 
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [51]:
def collate_fn(batch):
    imgs, labels_encoded, widths = zip(*batch)  
    max_w = max([img.size(2) for img in imgs])

    padded_images = []
    for img in imgs:
        pad_w = max_w - img.size(2)
        padded = torch.nn.functional.pad(img, (0, pad_w)) 
        padded_images.append(padded)

    imgs = torch.stack(padded_images)

    label_lens = torch.tensor([len(l) for l in labels_encoded], dtype=torch.long)
    labels = torch.cat(labels_encoded)
    widths = torch.tensor(widths, dtype=torch.long)  
    
    return imgs, labels, label_lens, widths  

In [52]:
dataset = WordImageDataset(
    image_dir='./Data/images',
    label_file="./Data/labels.txt",
    transform=transform
)

In [53]:
def compute_feature_length(input_width):
    return (input_width // 4) - 1

In [54]:
train_size = int(SPLIT_RATIO * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [55]:
# Check sequence lengths
print("Checking sequence lengths...")
for i in range(min(3, len(dataset))):
    img, label, width = dataset[i]
    seq_len = compute_feature_length(width)
    label_len = len(label)
    print(f"Sample {i}: Width={width}, SeqLen={seq_len}, LabelLen={label_len}")
    if seq_len <= label_len:
        print(f"  WARNING: Increase IMG_HEIGHT or adjust compute_feature_length")

Checking sequence lengths...
Sample 0: Width=96, SeqLen=23, LabelLen=8
Sample 1: Width=84, SeqLen=20, LabelLen=9
Sample 2: Width=122, SeqLen=29, LabelLen=8


In [56]:
class BidirectionalLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super(BidirectionalLSTM, self).__init__()
        self.rnn = nn.LSTM(input_size, hidden_size, bidirectional=True, batch_first=True)
        self.linear = nn.Linear(hidden_size * 2, output_size)

    def forward(self, input):
        """
        input : visual feature [batch_size x T x input_size]
        output : contextual feature [batch_size x T x output_size]
        """
        try: # multi gpu needs this
            self.rnn.flatten_parameters()
        except: # quantization doesn't work with this
            pass
        recurrent, _ = self.rnn(input)  # batch_size x T x input_size -> batch_size x T x (2*hidden_size)
        output = self.linear(recurrent)  # batch_size x T x output_size
        return output

class VGG_FeatureExtractor(nn.Module):

    def __init__(self, input_channel, output_channel=256):
        super(VGG_FeatureExtractor, self).__init__()
        self.output_channel = [int(output_channel / 8), int(output_channel / 4),
                               int(output_channel / 2), output_channel]
        self.ConvNet = nn.Sequential(
            nn.Conv2d(input_channel, self.output_channel[0], 3, 1, 1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.output_channel[0], self.output_channel[1], 3, 1, 1), nn.ReLU(True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(self.output_channel[1], self.output_channel[2], 3, 1, 1), nn.ReLU(True),
            nn.Conv2d(self.output_channel[2], self.output_channel[2], 3, 1, 1), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(self.output_channel[2], self.output_channel[3], 3, 1, 1, bias=False),
            nn.BatchNorm2d(self.output_channel[3]), nn.ReLU(True),
            nn.Conv2d(self.output_channel[3], self.output_channel[3], 3, 1, 1, bias=False),
            nn.BatchNorm2d(self.output_channel[3]), nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),
            nn.Conv2d(self.output_channel[3], self.output_channel[3], 2, 1, 0), nn.ReLU(True))

    def forward(self, input):
        return self.ConvNet(input)

class CRNN(nn.Module):

    def __init__(self, input_channel=1, output_channel=512, hidden_size=256, num_class=len(CHARS)+1):
        super(CRNN, self).__init__()
        """ FeatureExtraction """
        self.FeatureExtraction = VGG_FeatureExtractor(input_channel, output_channel)
        self.FeatureExtraction_output = output_channel
        self.AdaptiveAvgPool = nn.AdaptiveAvgPool2d((None, 1))

        """ Sequence modeling"""
        self.SequenceModeling = nn.Sequential(
            BidirectionalLSTM(self.FeatureExtraction_output, hidden_size, hidden_size),
            BidirectionalLSTM(hidden_size, hidden_size, hidden_size))
        self.SequenceModeling_output = hidden_size

        """ Prediction """
        self.Prediction = nn.Linear(self.SequenceModeling_output, num_class)


    def forward(self, input):
        """ Feature extraction stage """
        visual_feature = self.FeatureExtraction(input)
        visual_feature = self.AdaptiveAvgPool(visual_feature.permute(0, 3, 1, 2))
        visual_feature = visual_feature.squeeze(3)

        """ Sequence modeling stage """
        contextual_feature = self.SequenceModeling(visual_feature)

        """ Prediction stage """
        prediction = self.Prediction(contextual_feature.contiguous())
        prediction = prediction.permute(1, 0, 2)

        return prediction

In [57]:
def greedy_decoder(output):
    output = output.softmax(2)
    max_indices = output.argmax(2).permute(1, 0)
    decoded = []
    for indices in max_indices:
        s = ""
        prev = 0
        for i in indices:
            if i != prev and i != 0:
                s += idx2char[i.item()]
            prev = i
        decoded.append(s)
    return decoded

In [58]:
model = CRNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
ctc_loss = nn.CTCLoss(blank=0, zero_infinity=True)  # avoid infinite loss values

In [59]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    batch_count = 0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for imgs, labels, label_lens, widths in progress_bar:
        imgs = imgs.to(device)
        labels = labels.to(device)
        label_lens = label_lens.to(device)

        # Skip batches with empty labels
        if labels.numel() == 0:
            continue

        optimizer.zero_grad()
        outputs = model(imgs)  # T x B x C
        log_probs = outputs.log_softmax(2)  # Apply log-softmax before CTCLoss
        
        # Get ACTUAL sequence length from model output
        actual_seq_len = outputs.size(0)
        batch_size = outputs.size(1)
        
        # Input lengths: actual sequence from model
        input_lengths = torch.full(
            size=(batch_size,),
            fill_value=actual_seq_len,
            dtype=torch.long,
            device=device
        )
        
        # CTC requirement: input_length > target_length
        # Clamp labels to ensure this constraint
        clamped_label_lens = torch.clamp(label_lens, max=actual_seq_len - 1)
        
        try:
            loss = criterion(log_probs, labels, input_lengths, clamped_label_lens)
            
            # Only update if loss is valid (not NaN or Inf)
            if torch.isfinite(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
                batch_count += 1
            else:
                print(f"⚠️  Invalid loss detected (NaN/Inf), skipping batch")
                
        except RuntimeError as e:
            print(f"⚠️  CTC Loss error: {e}")
            continue
        
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}' if torch.isfinite(loss) else 'NaN'})
    
    return total_loss / max(batch_count, 1)

In [60]:
def evaluate_model(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    batch_count = 0
    with torch.no_grad():
        progress_bar = tqdm(dataloader, desc="Validation", leave=False)
        for imgs, labels, label_lens, widths in progress_bar:
            imgs = imgs.to(device)
            labels = labels.to(device)
            label_lens = label_lens.to(device)

            # Skip batches with empty labels
            if labels.numel() == 0:
                continue

            outputs = model(imgs)
            log_probs = outputs.log_softmax(2)  # Apply log-softmax before CTCLoss
            
            # Get ACTUAL sequence length from model output
            actual_seq_len = outputs.size(0)
            batch_size = outputs.size(1)
            
            # Input lengths: actual sequence from model
            input_lengths = torch.full(
                size=(batch_size,),
                fill_value=actual_seq_len,
                dtype=torch.long,
                device=device
            )
            
            # CTC requirement: input_length > target_length
            clamped_label_lens = torch.clamp(label_lens, max=actual_seq_len - 1)
            
            try:
                loss = criterion(log_probs, labels, input_lengths, clamped_label_lens)
                
                if torch.isfinite(loss):
                    total_loss += loss.item()
                    batch_count += 1
                else:
                    print(f"⚠️  Invalid loss detected (NaN/Inf)")
                    
            except RuntimeError as e:
                print(f"⚠️  CTC Loss error: {e}")
                continue
            
            progress_bar.set_postfix({'loss': f'{loss.item():.4f}' if torch.isfinite(loss) else 'NaN'})

    return total_loss / max(batch_count, 1)

In [61]:
# Detailed batch diagnostics - find the real problem!
test_imgs, test_labels, test_label_lens, test_widths = next(iter(train_loader))
print(f"=== BATCH DIAGNOSTICS ===")
print(f"Image shape: {test_imgs.shape} (Batch, Channels, Height, Width)")
print(f"Resized widths: min={test_widths.min()}, max={test_widths.max()}, mean={test_widths.float().mean():.1f}")
print(f"Label counts: total={test_labels.numel()}, individual lengths min={test_label_lens.min()}, max={test_label_lens.max()}")

# Extra diagnostic: check label token values to ensure we don't use 0 (reserved for blank)
label_tokens = torch.unique(test_labels)
print(f"Label token range: min={label_tokens.min().item()}, max={label_tokens.max().item()}")
print(f"Expected classes (including blank): {len(CHARS)+1}")
if label_tokens.min().item() == 0:
    print("❌ Labels contain 0 which collides with CTC blank. Ensure text_to_labels offsets indices by +1.")
else:
    print(f"✓ Labels correctly start at 1 (blank reserved for 0)")

# Forward pass to see model output
with torch.no_grad():
    outputs = model(test_imgs.to(device))
    seq_len = outputs.size(0)
    print(f"\nModel output shape: {outputs.shape} (Time, Batch, Classes)")
    print(f"Sequence length: {seq_len}")
    print(f"Max label length in batch: {test_label_lens.max()}")
    
    if test_label_lens.max() >= seq_len:
        print(f"❌ PROBLEM: Labels longer than or equal to sequence length!")
        print(f"   Need: max_label_len < sequence_len")
        print(f"   Have: {test_label_lens.max()} >= {seq_len}")
    else:
        print(f"✓ CTC constraint satisfied: {test_label_lens.max()} < {seq_len}")

# Check for suspicious samples
print(f"\nLabel length distribution:")
for i, label_len in enumerate(test_label_lens):
    width = test_widths[i].item()
    print(f"  Sample {i}: width={width}, label_len={label_len}, seq_len would be={(width//4)-1}")

=== BATCH DIAGNOSTICS ===
Image shape: torch.Size([128, 1, 32, 122]) (Batch, Channels, Height, Width)
Resized widths: min=8, max=122, mean=49.3
Label counts: total=367, individual lengths min=1, max=10
Label token range: min=6, max=94
Expected classes (including blank): 95
✓ Labels correctly start at 1 (blank reserved for 0)

Model output shape: torch.Size([29, 128, 95]) (Time, Batch, Classes)
Sequence length: 29
Max label length in batch: 10
✓ CTC constraint satisfied: 10 < 29

Label length distribution:
  Sample 0: width=21, label_len=1, seq_len would be=4
  Sample 1: width=72, label_len=3, seq_len would be=17
  Sample 2: width=49, label_len=2, seq_len would be=11
  Sample 3: width=21, label_len=1, seq_len would be=4
  Sample 4: width=59, label_len=6, seq_len would be=13
  Sample 5: width=50, label_len=2, seq_len would be=11
  Sample 6: width=34, label_len=2, seq_len would be=7
  Sample 7: width=16, label_len=1, seq_len would be=3
  Sample 8: width=34, label_len=1, seq_len would be=7

In [62]:
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs):
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')

    for epoch in range(epochs):
        print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)
        val_loss = evaluate_model(model, val_loader, criterion)

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "./best_model.pth")
            print(f"Best model saved at epoch {epoch+1} with validation loss: {best_val_loss:.4f}")
        
    print("Training Completed")
    
    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train_model(model, train_loader, val_loader, optimizer, ctc_loss, EPOCHS)


Epoch [1/3]


Epoch 1/3, Train Loss: 4.5078, Val Loss: 3.5971
Best model saved at epoch 1 with validation loss: 3.5971

Epoch [2/3]


Training:  43%|████▎     | 170/391 [10:04<12:09,  3.30s/it, loss=3.4512]

In [ ]:
MODEL_PATH = "./best_model.pth"

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(train_losses, label='Train Loss', color='blue')
ax1.set_title('Training Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(val_losses, label='Validation Loss', color='orange')
ax2.set_title('Validation Loss')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, map_location=torch.device('cpu')))
model.eval()

img, label, width = dataset[0]  # Dataset now returns 3 values
with torch.no_grad():
    pred = model(img.unsqueeze(0).to(device))
    pred_text = greedy_decoder(pred)[0]

gt_text = "".join([idx2char[i.item()] for i in label])

print("GT :", gt_text)
print("PRED:", pred_text)

In [ ]:
model.eval()
for _ in range(10):
    idx = random.randint(0, len(dataset)-1)
    img, label, width = dataset[idx]  # Dataset now returns 3 values

    with torch.no_grad():
        pred = model(img.unsqueeze(0).to(device))
        pred_text = greedy_decoder(pred)[0]

    gt_text = "".join([idx2char[i.item()] for i in label])
    print(f"GT: {gt_text:15s} | PRED: {pred_text}")


In [ ]:
def calculate_cer(gt, pred):
    """Calculate Character Error Rate"""
    if len(gt) == 0:
        return 0 if len(pred) == 0 else 1
    
    # Simple approach: count different characters
    errors = sum(1 for i in range(max(len(gt), len(pred))) 
                 if i >= len(gt) or i >= len(pred) or gt[i] != pred[i])
    return errors / len(gt)

def calculate_wer(gt, pred):
    """Calculate Word Error Rate"""
    gt_words = gt.split()
    pred_words = pred.split()
    
    if len(gt_words) == 0:
        return 0 if len(pred_words) == 0 else 1
    
    errors = sum(1 for i in range(max(len(gt_words), len(pred_words))) 
                 if i >= len(gt_words) or i >= len(pred_words) or gt_words[i] != pred_words[i])
    return errors / len(gt_words)

def calculate_accuracy(gt, pred):
    """Calculate exact match accuracy"""
    return 1.0 if gt == pred else 0.0

# Evaluate on full validation set and compute metrics
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

cer_scores = []
wer_scores = []
accuracy_scores = []

print("Computing OCR Metrics on Validation Set...")
with torch.no_grad():
    for imgs, labels, label_lens, widths in tqdm(val_loader, desc="Evaluating"):  # 4 values from dataloader
        imgs = imgs.to(device)
        outputs = model(imgs)
        pred_texts = greedy_decoder(outputs)
        
        # Convert labels back to text
        idx = 0
        for i, label_len in enumerate(label_lens):
            gt_label = labels[idx:idx+label_len]
            gt_text = "".join([idx2char[l.item()] for l in gt_label])
            pred_text = pred_texts[i]
            
            cer_scores.append(calculate_cer(gt_text, pred_text))
            wer_scores.append(calculate_wer(gt_text, pred_text))
            accuracy_scores.append(calculate_accuracy(gt_text, pred_text))
            
            idx += label_len

# Compute average metrics
avg_cer = np.mean(cer_scores)
avg_wer = np.mean(wer_scores)
avg_accuracy = np.mean(accuracy_scores)

print(f"\n{'='*50}")
print(f"OCR Performance Metrics:")
print(f"{'='*50}")
print(f"Character Error Rate (CER): {avg_cer:.4f} ({avg_cer*100:.2f}%)")
print(f"Word Error Rate (WER):      {avg_wer:.4f} ({avg_wer*100:.2f}%)")
print(f"Accuracy:                    {avg_accuracy:.4f} ({avg_accuracy*100:.2f}%)")
print(f"{'='*50}")

In [ ]:
# Visualize OCR Metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Metrics Summary Bar Chart
metrics = ['CER', 'WER', 'Accuracy']
values = [avg_cer, avg_wer, avg_accuracy]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

ax = axes[0, 0]
bars = ax.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black', linewidth=2)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('OCR Performance Metrics Summary', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1)
for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.2%}', 
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# 2. CER Distribution
ax = axes[0, 1]
ax.hist(cer_scores, bins=30, color='#FF6B6B', alpha=0.7, edgecolor='black')
ax.axvline(avg_cer, color='red', linestyle='--', linewidth=2, label=f'Mean: {avg_cer:.4f}')
ax.set_xlabel('Character Error Rate', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Character Error Rate Distribution', fontsize=13, fontweight='bold')
ax.legend()

# 3. WER Distribution
ax = axes[1, 0]
ax.hist(wer_scores, bins=30, color='#4ECDC4', alpha=0.7, edgecolor='black')
ax.axvline(avg_wer, color='darkgreen', linestyle='--', linewidth=2, label=f'Mean: {avg_wer:.4f}')
ax.set_xlabel('Word Error Rate', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Word Error Rate Distribution', fontsize=13, fontweight='bold')
ax.legend()

# 4. Accuracy Distribution
ax = axes[1, 1]
unique, counts = np.unique(accuracy_scores, return_counts=True)
ax.bar(unique, counts, color='#45B7D1', alpha=0.7, edgecolor='black', width=0.3)
ax.set_xlabel('Match (0=False, 1=True)', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Exact Match Accuracy Distribution', fontsize=13, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['Incorrect', 'Correct'])
correct_count = int(np.sum(accuracy_scores))
total_count = len(accuracy_scores)
ax.text(0.5, max(counts)*0.9, f'{correct_count}/{total_count}', 
        ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"\nSample sizes:")
print(f"  Total predictions: {len(accuracy_scores)}")
print(f"  Correct predictions: {correct_count}")
print(f"  Incorrect predictions: {total_count - correct_count}")

In [ ]:
# Visualize Sample Predictions with Ground Truth
num_samples = 12
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

model.eval()
with torch.no_grad():
    sample_idx = 0
    for imgs, labels, label_lens, widths in val_loader:  # 4 values from dataloader
        if sample_idx >= num_samples:
            break
            
        imgs = imgs.to(device)
        outputs = model(imgs)
        pred_texts = greedy_decoder(outputs)
        
        # Convert labels back to text
        label_idx = 0
        for batch_idx in range(imgs.shape[0]):
            if sample_idx >= num_samples:
                break
                
            label_len = label_lens[batch_idx].item()
            gt_label = labels[label_idx:label_idx+label_len]
            gt_text = "".join([idx2char[l.item()] for l in gt_label])
            pred_text = pred_texts[batch_idx]
            
            # Display image
            ax = axes[sample_idx]
            img = imgs[batch_idx, 0].cpu().numpy()
            ax.imshow(img, cmap='gray')
            
            # Color code based on correctness
            is_correct = gt_text == pred_text
            color = 'green' if is_correct else 'red'
            
            title = f"GT: {gt_text}\nPred: {pred_text}"
            ax.set_title(title, fontsize=10, fontweight='bold', color=color)
            ax.axis('off')
            
            label_idx += label_len
            sample_idx += 1

plt.suptitle('Sample OCR Predictions (Green=Correct, Red=Incorrect)', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# # Load image
# image_path = "/content/drive/MyDrive/yoloimg.jpg"
# img = cv2.imread(image_path)

# if img is None:
#     raise FileNotFoundError(f"Could not load image from {image_path}. Please check the path and file integrity.")

# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# H, W, _ = img.shape


In [ ]:
# yolo_boxes = [
#     [0, 0.898804, 0.545356, 0.0538002, 0.0604752, 0.927195],
#     [0, 0.453032, 0.269978, 0.137489, 0.0734341, 0.940134],
#     [0, 0.106746, 0.689525, 0.0785653, 0.0701944, 0.941084],
#     [0, 0.567037, 0.566955, 0.087105, 0.0583153, 0.942859],
# ]

In [ ]:
# def yolo_to_xyxy(box, img_w, img_h):
#     _, x_c, y_c, w, h, conf = box

#     x_c *= img_w
#     y_c *= img_h
#     w *= img_w
#     h *= img_h

#     x1 = int(x_c - w / 2)
#     y1 = int(y_c - h / 2)
#     x2 = int(x_c + w / 2)
#     y2 = int(y_c + h / 2)

#     return x1, y1, x2, y2, conf

In [ ]:
# pixel_boxes = [yolo_to_xyxy(b, W, H) for b in yolo_boxes]

In [ ]:
# def sort_boxes_reading_order(boxes, line_threshold=0.05):
#     # boxes: (x1, y1, x2, y2, conf)
#     boxes = sorted(boxes, key=lambda b: (b[1], b[0]))

#     lines = []
#     current_line = []

#     for box in boxes:
#         if not current_line:
#             current_line.append(box)
#         else:
#             if abs(box[1] - current_line[-1][1]) < line_threshold * H:
#                 current_line.append(box)
#             else:
#                 lines.append(sorted(current_line, key=lambda b: b[0]))
#                 current_line = [box]

#     if current_line:
#         lines.append(sorted(current_line, key=lambda b: b[0]))

#     return [b for line in lines for b in line]


In [ ]:
# pixel_boxes = sort_boxes_reading_order(pixel_boxes)

In [ ]:
# cropped_images = []

# for x1, y1, x2, y2, conf in pixel_boxes:
#     crop = img[max(0,y1):min(H,y2), max(0,x1):min(W,x2)]
#     cropped_images.append(crop)

#     plt.imshow(crop)
#     plt.axis("off")
#     plt.show()

In [ ]:
# def preprocess_crop(crop):
#     pil_img = Image.fromarray(crop).convert("L")

#     pil_img = ResizeKeepAspectRatio(32)(pil_img)
#     tensor = transforms.ToTensor()(pil_img)
#     tensor = transforms.Normalize((0.5,), (0.5,))(tensor)

#     return tensor.unsqueeze(0)  # add batch dimension

In [ ]:
# model.eval()
# results = []

# with torch.no_grad():
#     for crop in cropped_images:
#         inp = preprocess_crop(crop).to(device)
#         pred = model(inp)
#         text = greedy_decoder(pred)[0]
#         results.append(text)

In [ ]:
# print("Recognized text:")
# for t in results:
#     print(t)

In [ ]:
# img_vis = img.copy()

# for (x1, y1, x2, y2, _), text in zip(pixel_boxes, results):
#     cv2.rectangle(img_vis, (x1,y1), (x2,y2), (0,255,0), 2)
#     cv2.putText(img_vis, text, (x1, y1-5),
#                 cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)

# plt.imshow(img_vis)
# plt.axis("off")
# plt.show()

In [ ]:
# yolo = YOLO("/content/drive/MyDrive/best.pt")